# Force-density cable

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

%pip install -q compas_fd==0.5.4

In [ ]:
from compas.datastructures import Graph
from compas.geometry import Pointcloud
from compas_fd.solvers import fd_numpy
from compas_notebook import Viewer
from compas_notebook.config import Config

## Cable graph

In [ ]:
nodes = [[0, 0, 0], [1, 0, 0], [2, 0, 0], [3, 0, 0], [4, 0, 0]]
edges = [(0, 1), (1, 2), (2, 3), (3, 4)]
fixed = [0, 4]
cable = Graph.from_nodes_and_edges(nodes, edges)

## Force density

In [ ]:
P = 1.0
q = 2.0
result = fd_numpy(
    vertices=nodes,
    fixed=fixed,
    edges=edges,
    forcedensities=[q] * len(edges),
    loads=[[0, 0, 0] if key in fixed else [0, 0, -P] for key in cable.nodes()],
)
for key, xyz in zip(cable.nodes(), result.vertices):
    cable.node_attributes(key, "xyz", xyz.tolist())
for edge, force in zip(edges, result.forces):
    cable.edge_attribute(edge, "force", float(force[0]))
result.vertices[:, 2], result.forces[:, 0]

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [2, -7, 2]
config.view.camera.target = [2, 0, -0.5]
viewer = Viewer(config=config)
viewer.scene.add(cable, color="#dc4b8d", nodesize=0.15)
viewer.scene.add(Pointcloud([cable.node_coordinates(key) for key in fixed]), color="#348f9e", pointsize=0.25)
viewer.show()